### Jupyter Notebook (Development)
This notebook is used for testing, debugging, and exploring ideas during development.
* Run code step by step
* Check outputs and data shapes
* Fix issues in real time

---

###  Python Script (Production)
The final logic is moved to `database_manager.py` for actual system use.
* Ready for integration into `main_pipeline.py`
* Faster and cleaner execution
* Reusable as a module by other team members

# Module 7: Database Management System
### Centralized Support for the Face Recognition Pipeline

This module works as the **Data Center** of the Face Recognition Attendance System. It manages student face embeddings and provides organized, analysis-ready data for other modules in the pipeline.

---

### Key Responsibilities:
* **Data Storage:** Save and load face embeddings securely in `face_embeddings.pkl`.
* **Pipeline Support:** Provide structured `X` (features) and `y` (labels) arrays for the classification and matching modules.
* **Student Management:** Handle full CRUD operations (Add, Read, Update, and Delete) for student records.
* **Validation:** Check data consistency and ensure correct embedding dimensions across the database.

In [34]:
import os
import pickle
import numpy as np

In [35]:
def get_db_path():
    candidates = [
        "outputs/database/face_embeddings.pkl",
        "../outputs/database/face_embeddings.pkl"
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    default = "../outputs/database/face_embeddings.pkl"
    os.makedirs(os.path.dirname(default), exist_ok=True)
    return default

### Load Database
Opens the vault. Returns an empty dict gracefully if the file doesn't exist yet — no crashes on first run.

In [23]:
def load_database():
    path = get_db_path()
    if not os.path.exists(path):
        return {}
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception as e:
        print(f"Failed to load database: {e}")
        return {}

### Save Database
Locks the vault back up. Every write operation calls this to make sure nothing is lost.

In [24]:
def save_database(db):
    path = get_db_path()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(db, f)

### Add Student
Enrolls a new student. Computes the mean embedding automatically and refuses to overwrite an existing entry.

In [25]:
def add_student(student_id, embeddings):
    db = load_database()
    if student_id in db:
        print(f"Student '{student_id}' already exists.")
        return
    db[student_id] = {
        "mean": np.mean(embeddings, axis=0),
        "all": list(embeddings)
    }
    save_database(db)

### Remove Student
Drops a student from the database cleanly — no orphaned data left behind.

In [26]:
def remove_student(student_id):
    db = load_database()
    if student_id not in db:
        print(f"Student '{student_id}' not found.")
        return
    del db[student_id]
    save_database(db)


### Update Student
Got new photos? This appends fresh embeddings to the existing ones and recalculates the mean on the fly.

In [27]:

def update_student(student_id, new_embeddings):
    db = load_database()
    if student_id not in db:
        print(f"Student '{student_id}' not found.")
        return
    db[student_id]["all"].extend(new_embeddings)
    db[student_id]["mean"] = np.mean(db[student_id]["all"], axis=0)
    save_database(db)


### Get Student
Pulls one student's full record — their mean vector and all individual embeddings.

In [28]:
def get_student(student_id):
    db = load_database()
    if student_id not in db:
        print(f"Student '{student_id}' not found.")
        return None
    return db[student_id]

### List Students
Returns a clean sorted list of every enrolled student ID.

In [29]:
def list_students():
    db = load_database()
    return sorted(db.keys())

### Validate Database
Runs a health check on the entire database — catches missing keys, empty entries, and wrong embedding dimensions before they break something downstream.

In [30]:
def validate_database():
    db = load_database()
    issues = []
    for sid, entry in db.items():
        if "mean" not in entry or "all" not in entry:
            issues.append(f"'{sid}': missing keys")
            continue
        if len(entry["all"]) == 0:
            issues.append(f"'{sid}': empty embeddings")
        if entry["mean"].shape[0] != 512:
            issues.append(f"'{sid}': wrong embedding size {entry['mean'].shape}")
    return issues

### Prepare Data
The bridge to the classification module. Converts the database into `X` (features) and `y` (labels) arrays ready for the SVM.

In [31]:
def prepare_data(use_mean_only=False):
    db = load_database()
    if not db:
        return np.array([]), np.array([])

    X, y = [], []
    for student_id, entry in db.items():
        if use_mean_only:
            X.append(entry["mean"])
            y.append(student_id)
        else:
            for emb in entry["all"]:
                X.append(emb)
                y.append(student_id)

    return np.array(X), np.array(y)

### Demo — Database Summary
Proof of life: 40 students, 400 embeddings, zero issues.

In [32]:
students = list_students()
print(f"Students: {len(students)}")

issues = validate_database()
print(f"Issues: {issues if issues else 'None'}")

X, y = prepare_data()
print(f"X: {X.shape} | y: {y.shape}")

Students: 40
Issues: None
X: (400, 512) | y: (400,)


### Demo — CRUD in Action
Add → Update → Inspect → Remove. Full lifecycle test using dummy embeddings to make sure everything works end to end.

In [33]:
dummy = [np.random.randn(512) for _ in range(10)]

add_student("s_test", dummy)
update_student("s_test", [np.random.randn(512) for _ in range(5)])
print(get_student("s_test")["mean"].shape)
remove_student("s_test")

(512,)
